## Assignment 3 

### Task 1 - Data Ingestion

In [33]:
from pathlib import Path
import pandas as pd

In [ ]:


DATA_DIR = Path("/d/hpc/projects/FRI/bigdata/data/Taxi")

files = sorted(DATA_DIR.rglob("yellow_tripdata_*.parquet"))
print("Total yellow taxi parquet files:", len(files))
print("First few files:")
for f in files[:10]:
    print(f.name)


Total yellow taxi parquet files: 193
First few files:
yellow_tripdata_2009-01.parquet
yellow_tripdata_2009-02.parquet
yellow_tripdata_2009-03.parquet
yellow_tripdata_2009-04.parquet
yellow_tripdata_2009-05.parquet
yellow_tripdata_2009-06.parquet
yellow_tripdata_2009-07.parquet
yellow_tripdata_2009-08.parquet
yellow_tripdata_2009-09.parquet
yellow_tripdata_2009-10.parquet


In [6]:
files_by_year = {}
for f in files:
    year = f.stem.split("_")[-1][:4]
    files_by_year.setdefault(year, []).append(str(f))

print("Available years:", sorted(files_by_year.keys()))


Available years: ['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [7]:
selected_years = ["2019", "2020", "2021", "2022", "2023"]
for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    print(f"\n===== YEAR {year} =====")
    print("Number of columns:", len(df.columns))
    print(df.dtypes.astype(str))


===== YEAR 2019 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                      object
dtype: str

===== YEAR 2020 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count    

In [19]:
target_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]


In [20]:
def normalize_partition(pdf):
    pdf = pdf.copy()

    for col in target_columns:
        if col not in pdf.columns:
            pdf[col] = pd.NA

    pdf = pdf[target_columns]

    numeric_cols = [
        "VendorID", "passenger_count", "trip_distance", "RatecodeID",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "total_amount", "congestion_surcharge",
        "airport_fee"
    ]

    for col in numeric_cols:
        pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

    pdf["store_and_fwd_flag"] = pdf["store_and_fwd_flag"].astype("string")
    pdf["tpep_pickup_datetime"] = pd.to_datetime(pdf["tpep_pickup_datetime"], errors="coerce")
    pdf["tpep_dropoff_datetime"] = pd.to_datetime(pdf["tpep_dropoff_datetime"], errors="coerce")

    return pdf


In [22]:
meta = pd.DataFrame({
    "VendorID": pd.Series(dtype="float64"),
    "tpep_pickup_datetime": pd.Series(dtype="datetime64[ns]"),
    "tpep_dropoff_datetime": pd.Series(dtype="datetime64[ns]"),
    "passenger_count": pd.Series(dtype="float64"),
    "trip_distance": pd.Series(dtype="float64"),
    "RatecodeID": pd.Series(dtype="float64"),
    "store_and_fwd_flag": pd.Series(dtype="string"),
    "PULocationID": pd.Series(dtype="float64"),
    "DOLocationID": pd.Series(dtype="float64"),
    "payment_type": pd.Series(dtype="float64"),
    "fare_amount": pd.Series(dtype="float64"),
    "extra": pd.Series(dtype="float64"),
    "mta_tax": pd.Series(dtype="float64"),
    "tip_amount": pd.Series(dtype="float64"),
    "tolls_amount": pd.Series(dtype="float64"),
    "improvement_surcharge": pd.Series(dtype="float64"),
    "total_amount": pd.Series(dtype="float64"),
    "congestion_surcharge": pd.Series(dtype="float64"),
    "airport_fee": pd.Series(dtype="float64"),
    "year": pd.Series(dtype="int64"),
})


In [23]:
normalized_dfs = {}

for year in selected_years:
    monthly_dfs = []

    for file in files_by_year[year]:
        df_month = dd.read_parquet(file)
        df_month = df_month.map_partitions(normalize_partition, meta=meta.drop(columns=["year"]))
        monthly_dfs.append(df_month)

    df_year = dd.concat(monthly_dfs)
    df_year = df_year.assign(year=int(year))
    normalized_dfs[year] = df_year

    print(year, "normalized")



2019 normalized
2020 normalized
2021 normalized
2022 normalized
2023 normalized


In [26]:
OUTPUT_DIR = Path("/d/hpc/home/bv7063/bigdata/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for year in selected_years:
    out_dir = OUTPUT_DIR / f"csv_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)

    normalized_dfs[year].to_csv(
        str(out_dir / "part-*.csv"),
        index=False
    )

    print(f"{year} CSV written to {out_dir}")


2019 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2019
2020 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2020
2021 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2021
2022 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2022
2023 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2023


In [ ]:
# number of partitions per year
for year in selected_years:
    print(year, normalized_dfs[year].npartitions)


2019 12
2020 12
2021 12
2022 12
2023 12


In [30]:
## HDF5 VS CSV
one_year = "2023"
one_year_df = normalized_dfs[one_year]

one_year_csv_dir = OUTPUT_DIR / f"one_year_csv_{one_year}"
one_year_csv_dir.mkdir(parents=True, exist_ok=True)

one_year_pd = one_year_df.compute()
one_year_pd.to_hdf(
    OUTPUT_DIR / f"{one_year}.h5",
    key="taxi",
    mode="w",
    format="table"
)

one_year_df.to_csv(
    str(one_year_csv_dir / "part-*.csv"),
    index=False
)


['/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-00.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-01.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-02.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-03.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-04.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-05.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-06.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-07.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-08.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-09.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-10.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-11.csv']

In [ ]:
# combined parquet partion by year
combined_df = dd.concat([normalized_dfs[year] for year in selected_years])
combined_df = combined_df.repartition(partition_size="256MB")
optimized_parquet_dir = OUTPUT_DIR / "optimized_parquet"

combined_df.to_parquet(
    optimized_parquet_dir,
    engine="pyarrow",
    write_index=False,
    partition_on=["year"],
    row_group_size=100_000
)

# partion size means approximate size of each dask partion 
# row group size means within one dask partion / parquet file how are the rows goruped internally  
# dask partion and parquet file is not the same and can differ but durign writing one dask partion often becomes one parquest file

### Task 2 - Comparing Storage Formats


In [34]:
def get_size_bytes(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

def bytes_to_gb(x):
    return x / (1024**3)


In [44]:
# 5-year original parquet size
original_parquet_size = sum(
    Path(f).stat().st_size
    for year in selected_years
    for f in files_by_year[year]
)

# 5-year total CSV size
total_csv_5y = sum(
    get_size_bytes(OUTPUT_DIR / f"csv_{year}")
    for year in selected_years
)

# 5-year optimized parquet size
optimized_parquet_size = get_size_bytes(OUTPUT_DIR / "optimized_parquet")

comparison_five_years = pd.DataFrame([
    {"format": "Original parquet (5 years)", "size_bytes": original_parquet_size},
    {"format": "CSV total (5 years)", "size_bytes": total_csv_5y},
    {"format": "Optimized parquet (5 years)", "size_bytes": optimized_parquet_size},
])

comparison_five_years["size_gb"] = comparison_five_years["size_bytes"].apply(bytes_to_gb)
comparison_five_years = comparison_five_years.sort_values("size_gb")

print("Five-year comparison")
display(comparison_five_years)


Five-year comparison


,format,size_bytes,size_gb
0,Original parquet (5 years),3348556571,3.118586
2,Optimized parquet (5 years),4246213938,3.954595
1,CSV total (5 years),23447058209,21.836775


In [45]:
# 1-year comparison
one_year_original_parquet_size = sum(
    Path(f).stat().st_size for f in files_by_year[one_year]
)

one_year_csv_size = get_size_bytes(OUTPUT_DIR / f"one_year_csv_{one_year}")
one_year_hdf5_size = get_size_bytes(OUTPUT_DIR / f"{one_year}.h5")

comparison_one_year = pd.DataFrame([
    {"format": f"Original parquet ({one_year})", "size_bytes": one_year_original_parquet_size},
    {"format": f"CSV ({one_year})", "size_bytes": one_year_csv_size},
    {"format": f"HDF5 ({one_year})", "size_bytes": one_year_hdf5_size},
])

comparison_one_year["size_gb"] = comparison_one_year["size_bytes"].apply(bytes_to_gb)
comparison_one_year = comparison_one_year.sort_values("size_gb")

print(f"One-year comparison ({one_year})")
display(comparison_one_year)

One-year comparison (2023)


,format,size_bytes,size_gb
0,Original parquet (2023),635743618,0.592082
1,CSV (2023),4106528575,3.824503
2,HDF5 (2023),6296110531,5.863710


### Task 3 - todo 